# Ca Compound Analysis

This notebook queries the Materials Project for all Ca-containing compounds, computes the Voronoi polyhedron volume at each Ca site, and analyses the distribution to identify a target volume range.

## 1. Imports & Configuration

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymatgen

from mp_api.client import MPRester
from mp_api.client.core.client import MPRestError
from pymatgen.core import Element, Composition
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.inputs import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.transformations.standard_transformations import (
    OxidationStateDecorationTransformation,
    AutoOxiStateDecorationTransformation,
)
from pymatgen.analysis.energy_models import EnergyModel, EwaldElectrostaticModel
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.analysis.local_env import VoronoiNN
from itertools import combinations

In [ ]:
# Materials Project API key
api_key = 'dEomaClyhNd7JfX3NwpNUQXcEk8jo14c'

## 2. Function Definitions

In [ ]:
def voronoi_polyhedra_for_ca(compound, mpr):
    """
    Compute the average Voronoi polyhedron volume for Ca sites in a given compound.

    Parameters
    ----------
    compound : MPDataDoc
        A compound object returned by the MP API summary search.
    mpr : MPRester
        An active MPRester session.

    Returns
    -------
    float or None
        Average Voronoi polyhedron volume across all Ca sites,
        or None if no Ca sites are found or an error occurs.
    """
    try:
        structure = mpr.get_structure_by_material_id(compound.material_id)
        vnn = VoronoiNN()
        ca_volumes = []

        for i, site in enumerate(structure):
            if site.specie.symbol == "Ca":
                polyhedra = vnn.get_voronoi_polyhedra(structure, i)
                if polyhedra:
                    vol = sum(facet.get("volume", 0) for facet in polyhedra.values())
                else:
                    print(f"No polyhedra data for Ca site {i} in {compound.material_id}.")
                    vol = None
                if vol is not None:
                    ca_volumes.append(vol)

        return sum(ca_volumes) / len(ca_volumes) if ca_volumes else None

    except Exception as e:
        print(f"Error for {compound.material_id}: {e}")
        return None

## 3. Data Collection from Materials Project

Query all Ca-containing compounds, compute the Voronoi polyhedron volume at each Ca site, and save the results to CSV.

In [ ]:
fields = ["material_id", "formula_pretty", "elements", "energy_above_hull"]

with MPRester(api_key) as mpr:
    ca_compounds = mpr.materials.summary.search(elements=["Ca"], fields=fields)

    data = []
    for compound in ca_compounds:
        ca_vor_volume = voronoi_polyhedra_for_ca(compound, mpr)
        data.append({
            "material_id":      compound.material_id,
            "formula_pretty":   compound.formula_pretty,
            "elements":         compound.elements,
            "energy_above_hull": compound.energy_above_hull,
            "ca_vor_poly_volume": ca_vor_volume,
        })

df = pd.DataFrame(data)
df.to_csv("All_Ca_compounds.csv", index=False)
print(f"Saved {len(df)} entries to All_Ca_compounds.csv")

## 4. Load & Explore Data

### Ca compounds containing at least one transition metal and anion

Considered across all transition metals.

In [ ]:
ca_comp = pd.read_csv("All_Ca_Compounds.csv")
ca_comp.head()

In [ ]:
ca_comp.describe()

In [ ]:
ca_comp['ca_vor_poly_volume']

## 5. Statistical Analysis

### 5.1 Median and ±5 % target range

In [ ]:
median_val = ca_comp['ca_vor_poly_volume'].median()
print("Median:", median_val)

In [ ]:
pct = 0.05  # 5 %
lower = median_val * (1 - pct)
upper = median_val * (1 + pct)

subset = ca_comp[ca_comp['ca_vor_poly_volume'].between(lower, upper)]

print(f"Target range: {lower:.4f} – {upper:.4f}")
print(f"Compounds in range: {len(subset)}")

### 5.2 Middle 10 % quantile-based filtering

In [ ]:
col = "ca_vor_poly_volume"
fraction = 0.10
low_q  = 0.50 - fraction / 2
high_q = 0.50 + fraction / 2

low_val, high_val = ca_comp[col].quantile([low_q, high_q])
narrow_df = ca_comp[ca_comp[col].between(low_val, high_val)]

print(f"Quantile range: {low_val:.4f} – {high_val:.4f}")
print(f"Compounds in range: {len(narrow_df)}")

## 6. Filter Outliers

Remove entries with extremely large Voronoi volumes (> 50 Å³) to focus on physically reasonable Ca environments.

In [ ]:
ca_comp_c = ca_comp[ca_comp['ca_vor_poly_volume'] < 50]
print(f"Retained {len(ca_comp_c)} / {len(ca_comp)} entries after filtering.")
ca_comp_c.describe()

## 7. Visualization

In [ ]:
sns.set_theme(
    context="notebook",
    style="white",
    palette="flare",
    font="sans-serif",
    font_scale=1,
    color_codes=True,
)